# xgbGlobal Predictions 1959–2024

This notebook covers the full prediction and visualization pipeline for **xgbGlobal_sigHail** run over 1959–2024. It is organized into four parts:

1. **Shared Setup** — imports, constants, and reusable helper functions used by all subsequent sections.
2. **Prediction** — year-by-year generation and saving of predictions (global grid, CONUS, Europe).
3. **Frequency Maps** — mean annual significant-hail frequency maps (global overview + regional insets).
4. **Time Series** — annual hail event totals over time for global, CONUS, Europe, and selected cities.

---


## Part 1 — Shared Setup

### 1.1 Imports

All libraries used across the full notebook are imported here. Running this cell once is sufficient before executing any section below.


In [ ]:
from pathlib import Path
from datetime import datetime
import calendar

import numpy as np
import pandas as pd
from netCDF4 import Dataset
import xgboost as xgb

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle, ConnectionPatch
import cartopy.crs as ccrs
import cartopy.feature as cfeature


### 1.2 Global Constants

Defines the study period, missing years, predictor variable lists, and base file-system paths shared across all prediction and plotting sections. Adjust `fold_to_use`, `thr_bin`, and paths here to change the run configuration globally.


In [ ]:
# ── Study period ─────────────────────────────────────────────────────────────
START_DAY     = datetime(1959, 1, 1, 0)
STOP_DAY      = datetime(2024, 12, 31, 23)
MISSING_YEARS = [1966, 1979, 1999, 2003]

# ── Model / prediction settings ───────────────────────────────────────────────
FOLD        = 0          # fold used for prediction and plotting
THR_BIN     = 0.5        # binarisation threshold
CHUNK_DAYS  = 60         # days per prediction chunk (global); 180 for regional
SAVE_DAILY_PROB = False  # set True to save daily probabilities
SAVE_DAILY_BIN  = False  # set True to save daily binary fields

# ── Predictor variables ───────────────────────────────────────────────────────
ALL_MODEL_VARS = [
    "CAPEmax", "SRH03", "VS03", "FLH",
    "CINmax", "SRH06", "VS06", "DewT",
    "TotalTotals", "RH850", "RH500",
]
MODEL_VARS = [
    "CAPEmax", "SRH03", "VS03", "FLH",
    "CINmax", "VS06", "DewT",
    "TotalTotals",
]
SEL_IDX = [ALL_MODEL_VARS.index(v) for v in MODEL_VARS]
IX_VS03 = MODEL_VARS.index("VS03")
IX_VS06 = MODEL_VARS.index("VS06")

# ── Base paths ────────────────────────────────────────────────────────────────
BASE_CUMULUS = Path("/nfs/cumulus/highres_nobackup/agebhardt")
BASE_CLUSTER = Path("/cluster/home/agebhardt")
RUN_DATE     = "xgbGlobal_sigHail/final_dall"

PATHS = {
    # ERA5 predictor directories
    "era_global": BASE_CUMULUS / "e5_hailpredictors",
    "era_conus":  BASE_CUMULUS / "e5_hailpredictors_conus",
    "era_eu":     BASE_CUMULUS / "e5_hailpredictors_eu",
    "era_invar":  BASE_CUMULUS / "e5_data_processing" / "e5_invariant",

    # Observation directories
    "noaa_obs": BASE_CUMULUS / "hail_observations" / "SPC_data_griddedERA",
    "essl_obs": BASE_CUMULUS / "hail_observations" / "ESSL_data_griddedERA",

    # Model
    "models": BASE_CLUSTER / "models" / RUN_DATE,

    # Prediction output directories
    "pred_global": BASE_CUMULUS / "preds_1959-2024_xgbGlobal" / "preds_global" / f"fold{FOLD}_dall",
    "pred_us":     BASE_CUMULUS / "preds_1959-2024_xgbGlobal" / "preds_US"     / f"fold{FOLD}_dall",
    "pred_eu":     BASE_CUMULUS / "preds_1959-2024_xgbGlobal" / "preds_EU"     / f"fold{FOLD}_dall",

    # Plot output
    "plots": BASE_CLUSTER / "plots" / "preds_1959-2024_xgbGlobal",
}

# Region bounding boxes
EXTENTS = {
    "US": [-127, -66, 24, 50],   # [lon_min, lon_max, lat_min, lat_max]
    "EU": [-10,   35, 35, 65],
}

# Region-specific predictor / prediction config
REGION_CFG = {
    "US": {
        "era_dir":       PATHS["era_conus"],
        "pred_dir":      PATHS["pred_us"],
        "pred_prefix":   "pred_US",
        "infile_prefix": "ERA5_new_predictors_conus",
    },
    "EU": {
        "era_dir":       PATHS["era_eu"],
        "pred_dir":      PATHS["pred_eu"],
        "pred_prefix":   "pred_EU",
        "infile_prefix": "ERA5_new_predictors_eu",
    },
}

print("Selected predictor indices:", SEL_IDX, "=>", [ALL_MODEL_VARS[i] for i in SEL_IDX])
for p in [PATHS["pred_global"], PATHS["pred_us"], PATHS["pred_eu"], PATHS["plots"]]:
    p.mkdir(parents=True, exist_ok=True)


### 1.3 Shared Helper Functions

All reusable utility functions are defined here. They are called by the prediction, mapping, and time-series sections below, so this cell must be run first.


In [ ]:
# ── Time / year utilities ─────────────────────────────────────────────────────

def filtered_years(start_year, end_year, missing_years=MISSING_YEARS):
    """Return integer year array excluding missing years."""
    years = np.arange(start_year, end_year + 1, dtype=int)
    return years[~np.isin(years, missing_years)]


def filtered_daily_dates(start_year, end_year, missing_years=MISSING_YEARS):
    """Return daily DatetimeIndex excluding missing years."""
    dates = pd.date_range(datetime(start_year, 1, 1), end=datetime(end_year, 12, 31), freq="D")
    return dates[~dates.year.isin(missing_years)]


# ── Grid utilities ────────────────────────────────────────────────────────────

def ensure_2d_latlon(lat, lon):
    """Broadcast 1-D lat/lon arrays to 2-D grids if needed."""
    lat = np.asarray(lat)
    lon = np.asarray(lon)
    if lat.ndim == 1 and lon.ndim == 1:
        return (
            np.tile(lat[:, None], (1, lon.size)),
            np.tile(lon[None, :], (lat.size, 1)),
        )
    if lat.ndim == 2 and lon.ndim == 2:
        return lat, lon
    raise RuntimeError(f"Unexpected lat/lon shapes: {lat.shape}, {lon.shape}")


def load_sample_grid(predictor_dir, years, file_prefix):
    """Load lat2d/lon2d from the first available predictor .npz file."""
    for year in years:
        path = predictor_dir / f"{file_prefix}_{int(year)}.npz"
        if path.is_file():
            data = np.load(path)
            lat = data.get("rgrLat")
            lon = data.get("rgrLon")
            if lat is None or lon is None:
                raise RuntimeError(f"{path.name} lacks rgrLat / rgrLon")
            lat2d, lon2d = ensure_2d_latlon(lat, lon)
            print(f"Grid loaded from {path.name}: shape={lat2d.shape}")
            return lat2d.astype(np.float32), lon2d.astype(np.float32)
    raise FileNotFoundError(f"No predictor file found in {predictor_dir}")


def nearest_grid_idx(lat2d, lon2d, lat0, lon0):
    """Return (iy, ix) of the nearest grid point to (lat0, lon0)."""
    dist2 = (lat2d - lat0) ** 2 + (lon2d - lon0) ** 2
    return np.unravel_index(np.nanargmin(dist2), dist2.shape)


def get_3x3_slice(iy, ix, ny, nx):
    """Return (row_slice, col_slice) for a 3×3 box centred on (iy, ix)."""
    return (
        slice(max(0, iy - 1), min(ny, iy + 2)),
        slice(max(0, ix - 1), min(nx, ix + 2)),
    )


# ── Model I/O & prediction ────────────────────────────────────────────────────

def load_booster(model_path):
    """Load and return an XGBoost Booster from a .json file."""
    model_path = Path(model_path)
    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}")
    booster = xgb.Booster()
    booster.load_model(str(model_path))
    print("Loaded model:", model_path.name)
    return booster


def prepare_predictors(rgr_era_vars):
    """Apply abs() to VS03/VS06 (same transform as in training)."""
    x = rgr_era_vars.copy()
    x[:, IX_VS03, :, :] = np.abs(x[:, IX_VS03, :, :])
    x[:, IX_VS06, :, :] = np.abs(x[:, IX_VS06, :, :])
    return x


def predict_chunked(booster, x_grid, chunk_days=CHUNK_DAYS):
    """
    Predict over a 4-D predictor array (T, F, H, W) in time chunks.
    Returns y_prob (T, H, W).
    """
    t_size, n_feat, n_lat, n_lon = x_grid.shape
    y_prob = np.full((t_size, n_lat, n_lon), np.nan, dtype=np.float32)

    for t0 in range(0, t_size, chunk_days):
        t1 = min(t0 + chunk_days, t_size)
        x_chunk = np.moveaxis(x_grid[t0:t1], 1, -1)          # (T, H, W, F)
        x_flat  = x_chunk.reshape(-1, n_feat)
        dmat    = xgb.DMatrix(x_flat, missing=np.nan, feature_names=list(MODEL_VARS))
        pred    = booster.predict(dmat).astype(np.float32).reshape(t1 - t0, n_lat, n_lon)
        y_prob[t0:t1] = pred
        print(f"    chunk {t0:03d}:{t1:03d} done")

    return y_prob


# ── Observation loading ───────────────────────────────────────────────────────

def read_noaa_observations(noaa_dir, years, lon_min, lon_max, lat_min, lat_max):
    """Load NOAA SPC hail observations cropped to a bounding box."""
    hail_vars = ["Hail", "HailSize"]
    idx_lat = idx_lon = None
    total_days = sum(365 + int(calendar.isleap(int(y))) for y in years)
    obs = None
    dd = 0

    for y in years:
        nc_path = noaa_dir / f"SPC-Hail-StormReports_gridded-75km_{int(y)}.nc"
        with Dataset(str(nc_path), mode="r") as nc:
            lat1d = np.squeeze(nc.variables["lat"][:, 0]).astype(np.float32)
            lon1d = np.squeeze(nc.variables["lon"][0, :]).astype(np.float32)

            if idx_lat is None:
                idx_lat = np.where((lat1d >= lat_min) & (lat1d <= lat_max))[0]
                idx_lon = np.where((lon1d >= lon_min) & (lon1d <= lon_max))[0]
                ny, nx = idx_lat.size, idx_lon.size
                obs = np.zeros((2, total_days, ny, nx), dtype=np.float32)
                rgrLat = np.tile(lat1d[idx_lat, None], (1, nx))
                rgrLon = np.tile(lon1d[None, idx_lon], (ny, 1))
                print("NOAA grid:", ny, "x", nx,
                      "| lat", float(lat1d[idx_lat].min()), float(lat1d[idx_lat].max()),
                      "| lon", float(lon1d[idx_lon].min()), float(lon1d[idx_lon].max()))

            ylen = 365 + int(calendar.isleap(int(y)))
            for ii, vname in enumerate(hail_vars):
                data_full = np.squeeze(nc.variables[vname][:]).astype(np.float32)
                obs[ii, dd:dd + ylen] = data_full[:, idx_lat, :][:, :, idx_lon]
        dd += ylen

    return obs, rgrLat, rgrLon


def read_essl_observations(essl_dir, years, lon_min, lon_max, lat_min, lat_max):
    """Load ESSL hail observations cropped to a bounding box."""
    size_candidates = ["HailSize", "size", "Size"]
    idx_lat = idx_lon = size_varname = None
    total_days = sum(365 + int(calendar.isleap(int(y))) for y in years)
    obs = None
    dd = 0

    for y in years:
        nc_path = essl_dir / f"ESSL-Hail-StormReports_gridded-ERA5_{int(y)}.nc"
        with Dataset(str(nc_path), mode="r") as nc:
            if idx_lat is None:
                lat1d = np.squeeze(nc.variables["lat"][:, 0]).astype(np.float32)
                lon1d = np.squeeze(nc.variables["lon"][0, :]).astype(np.float32)
                lon1d = np.where(lon1d > 180, lon1d - 360, lon1d)
                idx_lat = np.where((lat1d >= lat_min) & (lat1d <= lat_max))[0]
                idx_lon = np.where((lon1d >= lon_min) & (lon1d <= lon_max))[0]
                size_varname = next((c for c in size_candidates if c in nc.variables), None)
                if size_varname is None:
                    raise KeyError(f"No hail size variable found. Tried: {size_candidates}")
                ny, nx = idx_lat.size, idx_lon.size
                obs = np.zeros((2, total_days, ny, nx), dtype=np.float32)
                rgrLat = np.tile(lat1d[idx_lat, None], (1, nx))
                rgrLon = np.tile(lon1d[None, idx_lon], (ny, 1))
                print("ESSL grid:", ny, "x", nx, "| size var:", size_varname,
                      "| lat", float(lat1d[idx_lat].min()), float(lat1d[idx_lat].max()),
                      "| lon", float(lon1d[idx_lon].min()), float(lon1d[idx_lon].max()))

            ylen = 365 + int(calendar.isleap(int(y)))
            hail_full = np.squeeze(nc.variables["Hail"][:]).astype(np.float32)
            size_full = np.squeeze(nc.variables[size_varname][:]).astype(np.float32)
            obs[0, dd:dd + ylen] = hail_full[:, idx_lat, :][:, :, idx_lon]
            obs[1, dd:dd + ylen] = size_full[:, idx_lat, :][:, :, idx_lon]
        dd += ylen

    return obs, rgrLat, rgrLon


def build_sig_hail(obs_2var, threshold_cm):
    """Convert (2, T, H, W) obs array to binary significant-hail (T, H, W)."""
    hail_bin  = obs_2var[0].astype(np.float32)
    hail_size = obs_2var[1].astype(np.float32)
    return ((hail_bin == 1) & np.isfinite(hail_size) & (hail_size >= threshold_cm)).astype(np.float32)


# ── Prediction file I/O ───────────────────────────────────────────────────────

def save_year_output(out_npz, year, y_prob, y_bin, mean_annual):
    """Save per-year prediction output; daily fields are optional (see constants)."""
    save_dict = {
        "year":             np.int32(year),
        "mean_annual":      mean_annual.astype(np.float32),
        "model_vars":       np.array(MODEL_VARS, dtype="U"),
        "threshold_binary": np.float32(THR_BIN),
        "fold":             np.int32(FOLD),
    }
    if SAVE_DAILY_PROB:
        save_dict["y_prob"] = y_prob.astype(np.float32)
    if SAVE_DAILY_BIN:
        save_dict["y_bin"] = y_bin.astype(np.float32)
    np.savez_compressed(out_npz, **save_dict)


def load_yearly_mean_annual(path):
    """Load 'mean_annual' field from a single yearly prediction file."""
    if not Path(path).is_file():
        return None
    data = np.load(path)
    if "mean_annual" not in data:
        raise RuntimeError(f"'mean_annual' missing in: {path}")
    return data["mean_annual"].astype(np.float32)


def load_prediction_timeseries(pred_dir, pred_prefix, years, fold):
    """
    Read saved yearly prediction files and return total modeled
    hail gridcell-events per year (summed over the whole domain).
    """
    yrs, vals = [], []
    for year in years:
        path = pred_dir / f"{pred_prefix}_{int(year)}_fold{fold}.npz"
        data = load_yearly_mean_annual(path)
        if data is None:
            print(f"[{pred_prefix} {year}] missing -> skip")
            continue
        ndays = 365 + int(calendar.isleap(int(year)))
        vals.append(float(np.nansum(data * ndays)))
        yrs.append(int(year))
    return np.array(yrs, dtype=int), np.array(vals, dtype=np.float32)


# ── Statistics ────────────────────────────────────────────────────────────────

def yearly_region_total(hail_3d, dates):
    """Sum total hail gridcell-events per year across (T, H, W) array."""
    years = np.unique(dates.year)
    vals  = np.array([np.nansum(hail_3d[dates.year == y]) for y in years], dtype=np.float32)
    return years.astype(int), vals


def fit_linear_trend(years, values):
    """Fit a linear trend; return (trend_line, slope_pct_per_decade)."""
    mask = np.isfinite(values)
    if mask.sum() < 2:
        return np.full(values.shape, np.nan), np.nan
    coef = np.polyfit(years[mask], values[mask], 1)
    trend = np.polyval(coef, years)
    mean_val = np.nanmean(values[mask])
    slope_pct = (coef[0] / mean_val) * 100 * 10 if mean_val != 0 else np.nan
    return trend, slope_pct


# ── Colormap ──────────────────────────────────────────────────────────────────

def make_frequency_cmap(max_val):
    """Return (cmap, norm, bounds) for mean annual hail frequency maps."""
    cmap = ListedColormap([
        "#f3efb4", "#dce9b8", "#b9ddb2", "#8fcfb5", "#63c0bf",
        "#4baed0", "#3f8fd1", "#3b6bc2", "#3247ad", "#1f2f88", "#081d58",
    ])
    base_bounds = [0.05, 0.2, 0.5, 1.0, 2.0, 4.0, 7.0]
    upper = (
        max_val if max_val <= 7
        else 8.0  if max_val <= 8
        else 10.0 if max_val <= 10
        else 12.0 if max_val <= 12
        else float(np.ceil(max_val))
    )
    bounds = [b for b in base_bounds if b < upper] + [float(upper)]
    norm = BoundaryNorm(bounds, cmap.N, extend="max")
    return cmap, norm, bounds


def format_cbar_ticklabels(bounds):
    return [f"{b:.2f}" if b < 1 else f"{b:.1f}" for b in bounds]


def mask_below_threshold(data, lower=0.05):
    out = np.ma.masked_invalid(data)
    return np.ma.masked_less(out, lower)


print("Helper functions loaded.")


---

## Part 2 — Prediction

### 2.1 Global Grid Prediction

Runs xgbGlobal on the full global ERA5 predictor grid (0.25°) year by year. Each year is saved as an independent `.npz` file containing `mean_annual` (and optionally daily probability / binary fields). Already completed years are skipped automatically.


In [ ]:
# ── Config specific to global prediction ─────────────────────────────────────
model_file = PATHS["models"] / f"xgbGlobal_sigHail_FINAL_2000-2022_fold{FOLD}.json"

years_all = filtered_years(START_DAY.year, STOP_DAY.year)
print(f"Years: {years_all[0]}–{years_all[-1]} | missing: {MISSING_YEARS}")

# Load sample predictor to determine grid shape
lat2d_glob, lon2d_glob = load_sample_grid(
    PATHS["era_global"], years_all, "ERA5_new_predictors"
)
ny_glob, nx_glob = lat2d_glob.shape

booster_global = load_booster(model_file)

# ── Main loop ─────────────────────────────────────────────────────────────────
n_done = n_skip_exist = n_skip_miss = n_skip_shape = 0

for year in years_all:
    year = int(year)
    infile  = PATHS["era_global"] / f"ERA5_new_predictors_{year}.npz"
    out_npz = PATHS["pred_global"] / f"pred_global_{year}_fold{FOLD}.npz"

    if out_npz.is_file():
        print(f"[{year}] exists -> skip")
        n_skip_exist += 1
        continue
    if not infile.is_file():
        print(f"[{year}] predictor missing -> skip")
        n_skip_miss += 1
        continue

    print(f"\n[{year}] loading predictors")
    data_tmp = np.load(infile)
    x_raw = data_tmp["rgrERAVarsyy"].astype(np.float32)[:, SEL_IDX, :, :]

    if x_raw.shape[2:] != (ny_glob, nx_glob):
        print(f"[{year}] grid mismatch {x_raw.shape[2:]} vs {(ny_glob, nx_glob)} -> skip")
        n_skip_shape += 1
        continue

    x_grid = prepare_predictors(x_raw)

    print(f"[{year}] predicting")
    y_prob = predict_chunked(booster_global, x_grid, chunk_days=CHUNK_DAYS)
    y_bin  = np.where(np.isnan(y_prob), np.nan, (y_prob >= THR_BIN).astype(np.float32))
    mean_annual = np.nansum(y_bin, axis=0) / y_bin.shape[0]

    save_year_output(out_npz, year, y_prob, y_bin, mean_annual)
    print(f"[{year}] saved -> {out_npz.name}")
    n_done += 1

print(f"\nGlobal prediction done | predicted={n_done} | "
      f"skipped_exist={n_skip_exist} | skipped_miss={n_skip_miss} | skipped_shape={n_skip_shape}")
print("Output directory:", PATHS["pred_global"])


### 2.2 Regional Prediction (CONUS & Europe)

Same resume-safe loop as above, but operating on the subsetted regional predictor files. Set `region` to `"US"` or `"EU"` and re-run this cell for each region.


In [ ]:
# ── Choose region: "US" or "EU" ──────────────────────────────────────────────
region = "EU"

rcfg      = REGION_CFG[region]
pred_dir  = rcfg["pred_dir"]
era_dir   = rcfg["era_dir"]
pred_pfx  = rcfg["pred_prefix"]
infile_pfx = rcfg["infile_prefix"]

model_file = PATHS["models"] / f"xgbGlobal_sigHail_FINAL_2000-2022_fold{FOLD}.json"

years_all = filtered_years(START_DAY.year, STOP_DAY.year)

# Load sample to determine grid shape
lat2d_reg, lon2d_reg = load_sample_grid(era_dir, years_all, infile_pfx)
ny_reg, nx_reg = lat2d_reg.shape

booster_reg = load_booster(model_file)

# ── Main loop ─────────────────────────────────────────────────────────────────
n_done = n_skip_exist = n_skip_miss = n_skip_shape = 0

for year in years_all:
    year    = int(year)
    infile  = era_dir  / f"{infile_pfx}_{year}.npz"
    out_npz = pred_dir / f"{pred_pfx}_{year}_fold{FOLD}.npz"

    if out_npz.is_file():
        print(f"[{region} {year}] exists -> skip")
        n_skip_exist += 1
        continue
    if not infile.is_file():
        print(f"[{region} {year}] predictor missing -> skip")
        n_skip_miss += 1
        continue

    print(f"\n[{region} {year}] loading predictors")
    data_tmp = np.load(infile)
    x_raw = data_tmp["rgrERAVarsyy"].astype(np.float32)[:, SEL_IDX, :, :]

    if x_raw.shape[2:] != (ny_reg, nx_reg):
        print(f"[{region} {year}] grid mismatch -> skip")
        n_skip_shape += 1
        continue

    x_grid = prepare_predictors(x_raw)

    print(f"[{region} {year}] predicting")
    y_prob = predict_chunked(booster_reg, x_grid, chunk_days=180)
    y_bin  = np.where(np.isnan(y_prob), np.nan, (y_prob >= THR_BIN).astype(np.float32))
    mean_annual = np.nansum(y_bin, axis=0) / y_bin.shape[0]

    save_year_output(out_npz, year, y_prob, y_bin, mean_annual)
    print(f"[{region} {year}] saved -> {out_npz.name}")
    n_done += 1

print(f"\n{region} prediction done | predicted={n_done} | "
      f"skipped_exist={n_skip_exist} | skipped_miss={n_skip_miss} | skipped_shape={n_skip_shape}")
print("Output directory:", pred_dir)


---

## Part 3 — Mean Annual Frequency Maps

### 3.1 Global Frequency Map

Loads all yearly `mean_annual` files for the global domain, averages across years, and plots a single global map of mean annual significant-hail days per grid cell.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
years = filtered_years(1959, 2024)
fold  = FOLD

outpath_global = PATHS["plots"] / (
    f"global_mean_annual_sig_hail_freq_{years[0]}-{years[-1]}_fold{fold}.png"
)

# ── Load grid + frequency field ───────────────────────────────────────────────
lat2d_glob, lon2d_glob = load_sample_grid(PATHS["era_global"], years, "ERA5_new_predictors")

yearly_fields = []
for year in years:
    path = PATHS["pred_global"] / f"pred_global_{year}_fold{fold}.npz"
    arr  = load_yearly_mean_annual(path)
    if arr is not None:
        yearly_fields.append(arr)

freq_global = np.nanmean(np.stack(yearly_fields, axis=0), axis=0) * 365.0

# ── Plot ──────────────────────────────────────────────────────────────────────
cmap, norm, bounds = make_frequency_cmap(float(np.nanmax(freq_global)))
freq_plot = mask_below_threshold(freq_global, lower=bounds[0])

fig = plt.figure(figsize=(15, 8))
ax  = plt.axes(projection=ccrs.Robinson())
ax.set_global()
ax.add_feature(cfeature.LAND,  facecolor="0.72", zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
ax.coastlines(linewidth=0.5, zorder=2)
ax.add_feature(cfeature.BORDERS, linewidth=0.25, alpha=0.35, zorder=2)

pcm = ax.pcolormesh(
    lon2d_glob, lat2d_glob, freq_plot,
    cmap=cmap, norm=norm, shading="auto",
    transform=ccrs.PlateCarree(), zorder=1,
)
cbar = plt.colorbar(pcm, ax=ax, orientation="vertical", shrink=0.82, pad=0.035, extend="max")
cbar.set_label("Mean significant hail days per year per grid cell", fontsize=14)
cbar.set_ticks(bounds)
cbar.set_ticklabels(format_cbar_ticklabels(bounds))
cbar.ax.tick_params(labelsize=12)

ax.set_title(
    f"Global Mean Annual Frequency of Significant Hail (≥4.4 cm)\n"
    f"xgbGlobal Final Model | {years[0]}–{years[-1]}",
    fontsize=16, pad=12,
)

fig.savefig(outpath_global, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath_global)


### 3.2 Regional Frequency Map (CONUS or Europe)

Plots the mean annual hail frequency for a single region with city markers. Set `region` to `"US"` or `"EU"`.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
region = "EU"  # "US" or "EU"
years  = filtered_years(1959, 2024)
fold   = FOLD

rcfg     = REGION_CFG[region]
extent   = EXTENTS[region]

cities = {
    # US cities
    "Oklahoma City": (-97.5164, 35.4676),
    "Kansas City":   (-94.5786, 39.0997),
    "Salt Lake City":(-111.891, 40.7608),
    # EU cities
    "Valencia": (-0.3763, 39.4699),
    "Paris":    ( 2.3522, 48.8566),
    "Zurich":   ( 8.5417, 47.3769),
    "Milan":    ( 9.1900, 45.4642),
    "Rome":     (12.4964, 41.9028),
    "Graz":     (15.4395, 47.0707),
    "Belgrade": (20.4489, 44.7866),
}
# Show only cities relevant to the chosen region
city_filter = {
    "US": ["Oklahoma City", "Kansas City", "Salt Lake City"],
    "EU": ["Valencia", "Paris", "Zurich", "Milan", "Rome", "Graz", "Belgrade"],
}
cities_to_plot = {k: v for k, v in cities.items() if k in city_filter[region]}

outpath_reg = PATHS["plots"] / (
    f"{region.lower()}_mean_annual_sig_hail_freq_{years[0]}-{years[-1]}_fold{fold}.png"
)

# ── Load grid + frequency field ───────────────────────────────────────────────
lat2d_reg, lon2d_reg = load_sample_grid(
    rcfg["era_dir"], years, rcfg["infile_prefix"]
)

yearly_fields = []
for year in years:
    path = rcfg["pred_dir"] / f"{rcfg['pred_prefix']}_{year}_fold{fold}.npz"
    arr  = load_yearly_mean_annual(path)
    if arr is not None:
        yearly_fields.append(arr)

freq_reg = np.nanmean(np.stack(yearly_fields, axis=0), axis=0) * 365.0

# ── Plot ──────────────────────────────────────────────────────────────────────
cmap, norm, bounds = make_frequency_cmap(float(np.nanmax(freq_reg)))
freq_plot = mask_below_threshold(freq_reg, lower=bounds[0])

fig = plt.figure(figsize=(12, 7))
ax  = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent(extent, crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND,  facecolor="0.85", zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
ax.coastlines(linewidth=0.5, zorder=2)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.35, zorder=2)
if region == "US":
    ax.add_feature(cfeature.STATES, linewidth=0.25, alpha=0.25, zorder=2)

pcm = ax.pcolormesh(
    lon2d_reg, lat2d_reg, freq_plot,
    cmap=cmap, norm=norm, shading="auto",
    transform=ccrs.PlateCarree(), zorder=1,
)
cbar = plt.colorbar(pcm, ax=ax, orientation="horizontal", pad=0.05, shrink=0.7, extend="max")
cbar.set_label("Mean significant hail days per year per grid cell", fontsize=12)
cbar.set_ticks(bounds)
cbar.set_ticklabels(format_cbar_ticklabels(bounds))

for city, (clon, clat) in cities_to_plot.items():
    ax.plot(clon, clat, "ko", markersize=4, transform=ccrs.PlateCarree(), zorder=5)
    ax.text(clon + 0.3, clat, city, fontsize=8,
            transform=ccrs.PlateCarree(), zorder=5)

region_label = "CONUS" if region == "US" else "Europe"
ax.set_title(
    f"{region_label} Mean Annual Frequency of Significant Hail (≥4.4 cm)\n"
    f"xgbGlobal (Fold {fold}) | {years[0]}–{years[-1]}",
    fontsize=13,
)

fig.savefig(outpath_reg, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath_reg)


### 3.3 Combined Map — Global Overview with CONUS & Europe Insets

Assembles the publication-ready figure: Robinson-projection global map with the full-resolution CONUS and Europe insets connected via boundary lines.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
fold_global, fold_us, fold_eu = FOLD, FOLD, FOLD
years  = filtered_years(1959, 2024)
extent_us = EXTENTS["US"]
extent_eu = EXTENTS["EU"]

outpath_combined = PATHS["plots"] / (
    f"combined_global_conus_europe_mean_annual_sig_hail_"
    f"{years[0]}-{years[-1]}_fg{fold_global}_fus{fold_us}_feu{fold_eu}.png"
)

# ── Load grids ────────────────────────────────────────────────────────────────
lat2d_glob, lon2d_glob = load_sample_grid(PATHS["era_global"], years, "ERA5_new_predictors")
lat2d_us,   lon2d_us   = load_sample_grid(PATHS["era_conus"], years, "ERA5_new_predictors_conus")
lat2d_eu,   lon2d_eu   = load_sample_grid(PATHS["era_eu"],   years, "ERA5_new_predictors_eu")

# ── Load frequency fields ─────────────────────────────────────────────────────
def _load_freq(pred_dir, prefix, years, fold):
    stacked = [f for year in years
               if (f := load_yearly_mean_annual(pred_dir / f"{prefix}_{year}_fold{fold}.npz")) is not None]
    return np.nanmean(np.stack(stacked, axis=0), axis=0) * 365.0

freq_global = _load_freq(PATHS["pred_global"], "pred_global", years, fold_global)
freq_us     = _load_freq(PATHS["pred_us"],     "pred_US",     years, fold_us)
freq_eu     = _load_freq(PATHS["pred_eu"],     "pred_EU",     years, fold_eu)

max_val = float(np.nanmax([np.nanmax(freq_global), np.nanmax(freq_us), np.nanmax(freq_eu)]))
cmap, norm, bounds = make_frequency_cmap(max_val)

freq_global_plot = mask_below_threshold(freq_global, bounds[0])
freq_us_plot     = mask_below_threshold(freq_us,     bounds[0])
freq_eu_plot     = mask_below_threshold(freq_eu,     bounds[0])

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 9))

# Main global map
ax_global = fig.add_axes([0.30, 0.23, 0.42, 0.60], projection=ccrs.Robinson())
ax_global.set_global()
ax_global.add_feature(cfeature.LAND,  facecolor="0.72", zorder=0)
ax_global.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
ax_global.coastlines(linewidth=0.5, zorder=2)
ax_global.add_feature(cfeature.BORDERS, linewidth=0.25, alpha=0.35, zorder=2)
pcm = ax_global.pcolormesh(
    lon2d_glob, lat2d_glob, freq_global_plot,
    cmap=cmap, norm=norm, shading="auto",
    transform=ccrs.PlateCarree(), zorder=1,
)

# Boxes on global map
for ext, col in [(extent_us, "black"), (extent_eu, "black")]:
    ax_global.add_patch(Rectangle(
        (ext[0], ext[2]), ext[1] - ext[0], ext[3] - ext[2],
        transform=ccrs.PlateCarree(),
        fill=False, edgecolor=col, linewidth=1.6, zorder=5,
    ))

# CONUS inset
ax_us = fig.add_axes([0.06, 0.65, 0.27, 0.24], projection=ccrs.PlateCarree())
ax_us.set_extent(extent_us, crs=ccrs.PlateCarree())
for feat in [cfeature.LAND, cfeature.OCEAN, cfeature.BORDERS, cfeature.STATES]:
    kwargs = {"facecolor": "0.85"} if feat in [cfeature.LAND] else {}
    kwargs = {"facecolor": "white"} if feat == cfeature.OCEAN else kwargs
    ax_us.add_feature(feat, linewidth=0.25, alpha=0.25, zorder=2, **kwargs)
ax_us.coastlines(linewidth=0.5, zorder=2)
ax_us.pcolormesh(lon2d_us, lat2d_us, freq_us_plot,
                 cmap=cmap, norm=norm, shading="auto",
                 transform=ccrs.PlateCarree(), zorder=1)
ax_us.set_title("CONUS", fontsize=11, pad=6)

# Europe inset
ax_eu = fig.add_axes([0.68, 0.65, 0.22, 0.24], projection=ccrs.PlateCarree())
ax_eu.set_extent(extent_eu, crs=ccrs.PlateCarree())
ax_eu.add_feature(cfeature.LAND,  facecolor="0.85", zorder=0)
ax_eu.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
ax_eu.coastlines(linewidth=0.5, zorder=2)
ax_eu.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.35, zorder=2)
ax_eu.pcolormesh(lon2d_eu, lat2d_eu, freq_eu_plot,
                 cmap=cmap, norm=norm, shading="auto",
                 transform=ccrs.PlateCarree(), zorder=1)
ax_eu.set_title("Europe", fontsize=11, pad=6)

# Connectors
for xyA, xyB, ax_inset in [
    ((extent_us[0], extent_us[3]), (1.00, 1.00), ax_us),
    ((extent_us[0], extent_us[2]), (1.00, 0.00), ax_us),
    ((extent_eu[1], extent_eu[3]), (0.00, 1.00), ax_eu),
    ((extent_eu[1], extent_eu[2]), (0.00, 0.00), ax_eu),
]:
    fig.add_artist(ConnectionPatch(
        xyA=xyA, coordsA=ccrs.PlateCarree()._as_mpl_transform(ax_global),
        xyB=xyB, coordsB=ax_inset.transAxes,
        color="black", lw=1.5, zorder=10,
    ))

# Colorbar
pos = ax_global.get_position()
cax = fig.add_axes([pos.x0 + pos.width * 0.2, pos.y0 - 0.06, pos.width * 0.6, 0.015])
cbar = fig.colorbar(pcm, cax=cax, orientation="horizontal", extend="max")
cbar.set_label("Average significant hail days per year per grid cell", fontsize=11)
cbar.set_ticks(bounds)
cbar.set_ticklabels(format_cbar_ticklabels(bounds))
cbar.ax.tick_params(labelsize=10)

fig.savefig(outpath_combined, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath_combined)


---

## Part 4 — Time Series

### 4.1 Load Observations & Predictions

Loads NOAA (CONUS) and ESSL (Europe) observations and the saved regional predictions. All time-series plots in sections 4.2–4.4 reuse these arrays, so run this cell first.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
THRESHOLD_CM  = 4.4
fold          = FOLD
PRED_START    = 1959
PRED_END      = 2024
EU_OBS_START  = 1979
EU_OBS_END    = 2022
PRED_COLOR    = "#d81b60"
LW            = 1.0

# Bounding boxes
us_lon_min, us_lon_max = -127.0, -66.0
us_lat_min, us_lat_max =   24.0,  50.0
eu_lon_min, eu_lon_max = -10.0,  35.0
eu_lat_min, eu_lat_max =  35.0,  65.0

# ── Time axes ─────────────────────────────────────────────────────────────────
years_pred   = filtered_years(PRED_START, PRED_END)
dates_pred   = filtered_daily_dates(PRED_START, PRED_END)
years_eu_obs = filtered_years(EU_OBS_START, EU_OBS_END)
dates_eu_obs = filtered_daily_dates(EU_OBS_START, EU_OBS_END)

print("Prediction years:", years_pred[0], "–", years_pred[-1], "| n =", len(years_pred))
print("EU obs years:    ", years_eu_obs[0], "–", years_eu_obs[-1], "| n =", len(years_eu_obs))

# ── Load observations ─────────────────────────────────────────────────────────
print("\nLoading NOAA observations ...")
rgrNOAAObs, rgrLatNOAA, rgrLonNOAA = read_noaa_observations(
    PATHS["noaa_obs"], years_pred,
    us_lon_min, us_lon_max, us_lat_min, us_lat_max,
)
print("rgrNOAAObs shape:", rgrNOAAObs.shape)

print("\nLoading ESSL observations ...")
rgrESSLObs, rgrLatESSL, rgrLonESSL = read_essl_observations(
    PATHS["essl_obs"], years_eu_obs,
    eu_lon_min, eu_lon_max, eu_lat_min, eu_lat_max,
)
print("rgrESSLObs shape:", rgrESSLObs.shape)

# ── Build significant-hail arrays ─────────────────────────────────────────────
sig_hail_us = build_sig_hail(rgrNOAAObs, THRESHOLD_CM)
sig_hail_eu = build_sig_hail(rgrESSLObs, THRESHOLD_CM)

# ── Annual observation totals ─────────────────────────────────────────────────
years_obs_us, obs_us = yearly_region_total(sig_hail_us, dates_pred)
years_obs_eu, obs_eu = yearly_region_total(sig_hail_eu, dates_eu_obs)

# ── Load prediction time series ───────────────────────────────────────────────
print("\nLoading regional predictions ...")
years_pred_us, pred_us = load_prediction_timeseries(
    PATHS["pred_us"], "pred_US", years_pred, fold
)
years_pred_eu, pred_eu = load_prediction_timeseries(
    PATHS["pred_eu"], "pred_EU", years_pred, fold
)

print("CONUS pred:  ", years_pred_us[0], "–", years_pred_us[-1])
print("Europe pred: ", years_pred_eu[0], "–", years_pred_eu[-1])
print("\nObs min/max  — CONUS:", np.nanmin(obs_us), "/", np.nanmax(obs_us),
      "  EU:", np.nanmin(obs_eu), "/", np.nanmax(obs_eu))
print("Pred min/max — CONUS:", np.nanmin(pred_us), "/", np.nanmax(pred_us),
      "  EU:", np.nanmin(pred_eu), "/", np.nanmax(pred_eu))


### 4.2 Regional Time Series — CONUS & Europe

2-panel figure with observations (black, left axis) and predictions (pink, right axis) for CONUS (top) and Europe (bottom), including a linear trend line for the predictions.


In [ ]:
outpath_ts = PATHS["plots"] / (
    f"timeseries_obs_vs_xgbglobal_conus_europe_fold{fold}_{PRED_START}-{PRED_END}.png"
)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), dpi=200, sharex=True)

for ax, region_label, years_obs, obs, years_pred_r, pred in [
    (axes[0], "CONUS",  years_obs_us, obs_us, years_pred_us, pred_us),
    (axes[1], "Europe", years_obs_eu, obs_eu, years_pred_eu, pred_eu),
]:
    axr = ax.twinx()
    ax.axvspan(2000, 2008, color="0.9",  zorder=0)
    ax.axvspan(2008, 2022, color="0.82", zorder=0)

    line_obs,  = ax.plot(years_obs,    obs,  color="black",      lw=LW)
    line_pred, = axr.plot(years_pred_r, pred, color=PRED_COLOR,   lw=LW)

    trend, slope_pct = fit_linear_trend(years_pred_r, pred)
    line_trend, = axr.plot(years_pred_r, trend, color=PRED_COLOR, lw=LW, linestyle="--")

    ax.set_title(region_label, fontsize=12)
    ax.tick_params(axis="y", colors="black",     labelsize=14)
    axr.tick_params(axis="y", colors=PRED_COLOR, labelsize=14)
    ax.grid(True, alpha=0.25)
    ax.legend(
        [line_obs, line_pred, line_trend],
        ["Observations", "Predictions", f"Trend: {slope_pct:.1f}% / decade"],
        loc="upper left", fontsize=14, frameon=False,
    )

axes[1].set_xlabel("Year", fontsize=14)
axes[1].set_xlim(PRED_START, PRED_END)
axes[1].set_xticks(np.arange(PRED_START, PRED_END + 1, 5))
axes[1].tick_params(axis="x", labelsize=14)

fig.suptitle(
    f"Observed and Modeled Significant Hail Events per Year\n"
    f"xgbGlobal (Fold {fold}) | CONUS and Europe",
    fontsize=14, y=0.98,
)
fig.text(0.015, 0.5, "Observed significant hail events per year",
         va="center", rotation="vertical", fontsize=14, color="black")
fig.text(0.985, 0.5, "Modeled significant hail events per year",
         va="center", rotation="vertical", fontsize=14, color=PRED_COLOR, ha="right")

fig.tight_layout(rect=[0.05, 0, 0.95, 0.96])
fig.savefig(outpath_ts, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath_ts)


### 4.3 Global Time Series (Predictions Only)

Plots the total number of modeled significant-hail gridcell-events per year over the whole global domain, with a linear trend annotation.


In [ ]:
# ── Load global predictions ───────────────────────────────────────────────────
years_pred_glob, pred_glob = load_prediction_timeseries(
    PATHS["pred_global"], "pred_global", years_pred, fold
)
print("Global pred:", years_pred_glob[0], "–", years_pred_glob[-1])

outpath_glob_ts = PATHS["plots"] / (
    f"timeseries_global_predictions_only_fold{fold}_{PRED_START}-{PRED_END}.png"
)

# ── Plot ──────────────────────────────────────────────────────────────────────
trend_glob, slope_glob = fit_linear_trend(years_pred_glob, pred_glob)

fig, ax = plt.subplots(figsize=(14, 5), dpi=200)
ax.axvspan(2000, 2022, color="0.9", zorder=0)

line_pred,  = ax.plot(years_pred_glob, pred_glob,  color=PRED_COLOR, lw=LW, label="Predictions")
line_trend, = ax.plot(years_pred_glob, trend_glob,  color=PRED_COLOR, lw=LW, linestyle="--",
                       label=f"Trend: {slope_glob:.1f}% / decade")

ax.set_title(f"Global Modeled Significant Hail Events per Year | xgbGlobal Fold {fold}",
             fontsize=12)
ax.set_ylabel("Modeled significant hail events per year", color=PRED_COLOR)
ax.set_xlabel("Year")
ax.tick_params(axis="y", colors=PRED_COLOR)
ax.set_xlim(PRED_START, PRED_END)
ax.set_xticks(np.arange(PRED_START, PRED_END + 1, 5))
ax.grid(True, alpha=0.25)
ax.legend(loc="upper left", fontsize=9, frameon=False)

fig.tight_layout()
fig.savefig(outpath_glob_ts, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath_glob_ts)


### 4.4 City-Level Time Series (3×3 Grid-Box Average)

Six-panel figure (3 CONUS + 3 European cities). For each city the annual hail count is averaged over a 3×3 grid-box neighbourhood to reduce point-level noise. Observations on the left axis, predictions + trend on the right axis.


In [ ]:
cities_us = {
    "Oklahoma City": {"lat": 35.4676, "lon": -97.5164},
    "Kansas City":   {"lat": 39.0997, "lon": -94.5786},
    "Dallas":        {"lat": 32.7767, "lon": -96.7970},
}
cities_eu = {
    "Graz":   {"lat": 47.0707, "lon": 15.4395},
    "Venice": {"lat": 45.4408, "lon": 12.3155},
    "Turin":  {"lat": 45.0703, "lon":  7.6869},
}

# ── Per-city helpers ──────────────────────────────────────────────────────────

def yearly_city_obs(sig_hail, dates, lat2d, lon2d, city):
    iy, ix = nearest_grid_idx(lat2d, lon2d, city["lat"], city["lon"])
    ys, xs = get_3x3_slice(iy, ix, sig_hail.shape[1], sig_hail.shape[2])
    years  = np.unique(dates.year)
    vals   = np.array([np.nansum(sig_hail[dates.year == y, ys, xs]) for y in years],
                      dtype=np.float32)
    return years.astype(int), vals


def yearly_city_pred(pred_dir, prefix, years, fold, lat2d, lon2d, city):
    iy, ix = nearest_grid_idx(lat2d, lon2d, city["lat"], city["lon"])
    ys, xs = get_3x3_slice(iy, ix, lat2d.shape[0], lat2d.shape[1])
    yrs, vals = [], []
    for year in years:
        path = pred_dir / f"{prefix}_{int(year)}_fold{fold}.npz"
        data = load_yearly_mean_annual(path)
        if data is None:
            continue
        ndays = 365 + int(calendar.isleap(int(year)))
        vals.append(float(np.nansum(data[ys, xs] * ndays)))
        yrs.append(int(year))
    return np.array(yrs, dtype=int), np.array(vals, dtype=np.float32)


# ── Build results ─────────────────────────────────────────────────────────────
city_results = []
for name, city in cities_us.items():
    y_obs,  obs_vals  = yearly_city_obs(sig_hail_us, dates_pred, rgrLatNOAA, rgrLonNOAA, city)
    y_pred, pred_vals = yearly_city_pred(PATHS["pred_us"], "pred_US", years_pred, fold,
                                         rgrLatNOAA, rgrLonNOAA, city)
    city_results.append({"city": name, "years_obs": y_obs, "obs": obs_vals,
                          "years_pred": y_pred, "pred": pred_vals})

for name, city in cities_eu.items():
    y_obs,  obs_vals  = yearly_city_obs(sig_hail_eu, dates_eu_obs, rgrLatESSL, rgrLonESSL, city)
    y_pred, pred_vals = yearly_city_pred(PATHS["pred_eu"], "pred_EU", years_pred, fold,
                                         rgrLatESSL, rgrLonESSL, city)
    city_results.append({"city": name, "years_obs": y_obs, "obs": obs_vals,
                          "years_pred": y_pred, "pred": pred_vals})

# ── Plot ──────────────────────────────────────────────────────────────────────
outpath_cities = PATHS["plots"] / (
    f"timeseries_city_level_3x3_fold{fold}_{PRED_START}-{PRED_END}.png"
)

fig, axes = plt.subplots(2, 3, figsize=(16, 7.5), dpi=200, sharex=True)
axes = axes.ravel()

for ax, result in zip(axes, city_results):
    axr = ax.twinx()
    ax.axvspan(2000, 2008, color="0.9",  zorder=0)
    ax.axvspan(2008, 2022, color="0.82", zorder=0)

    line_obs,  = ax.plot(result["years_obs"],  result["obs"],  color="black",    lw=1.0)
    line_pred, = axr.plot(result["years_pred"], result["pred"], color=PRED_COLOR, lw=1.0)
    trend, slope_pct = fit_linear_trend(result["years_pred"], result["pred"])
    line_trend, = axr.plot(result["years_pred"], trend, color=PRED_COLOR, lw=1.0, linestyle="--")

    ax.set_title(result["city"], fontsize=13)
    ax.grid(True, alpha=0.25)
    ax.tick_params(axis="y", labelsize=11, colors="black")
    axr.tick_params(axis="y", labelsize=11, colors=PRED_COLOR)
    ax.tick_params(axis="x", labelsize=11)
    ax.legend([line_obs, line_pred, line_trend],
              ["Observations", "Predictions", f"Trend: {slope_pct:.1f}% / decade"],
              fontsize=8, frameon=False, loc="upper left")

for ax in axes[3:]:
    ax.set_xlabel("Year", fontsize=12)

fig.text(0.01,  0.5, "Observed significant hail events per year",
         va="center", rotation="vertical", fontsize=13, color="black")
fig.text(0.99,  0.5, "Modeled significant hail events per year",
         va="center", rotation="vertical", fontsize=13, color=PRED_COLOR, ha="right")

fig.suptitle(f"City-Level Significant Hail Time Series | xgbGlobal Fold {fold}",
             fontsize=14, y=0.98)
fig.tight_layout(rect=[0.04, 0.02, 0.96, 0.95])
fig.savefig(outpath_cities, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath_cities)
